In [0]:
%sql
USE demo_snowday;

DROP TABLE IF EXISTS workspace.demo_snowday.landing_employee_cdf;



In [0]:
%sql
CREATE TABLE workspace.demo_snowday.landing_employee_cdf (
  id STRING,
  fullname STRING,
  sal STRING,
  doj STRING
) USING delta
TBLPROPERTIES (delta.enableChangeDataFeed = true);


In [0]:
%sql
INSERT INTO workspace.demo_snowday.landing_employee_cdf VALUES
('11','Adam','3000','2020-05-01'),
('13','Bob','200','2018-05-25'),
('14','King','100','2019-05-01');


num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
SELECT * FROM workspace.demo_snowday.landing_employee_cdf ORDER BY id;

id,fullname,sal,doj
11,Adam,3000,2020-05-01
13,Bob,200,2018-05-25
14,King,100,2019-05-01


In [0]:
%sql
-- step 1 to confirm bronze is ingesting data
SELECT id, _change_type, _commit_version, _commit_timestamp
FROM workspace.demo_snowday.bronze_employee_cdf
ORDER BY _commit_version;



id,_change_type,_commit_version,_commit_timestamp
11,insert,4,2026-03-02T20:47:56.000Z
13,insert,4,2026-03-02T20:47:56.000Z
15,insert,4,2026-03-02T20:47:56.000Z
13,update_postimage,5,2026-03-02T21:20:18.000Z
13,update_preimage,5,2026-03-02T21:20:18.000Z
13,update_preimage,6,2026-03-02T21:20:30.000Z
13,update_postimage,6,2026-03-02T21:20:30.000Z


In [0]:
%sql
-- step 2 Confirm Gold SCD1 has 3 rows
SELECT * 
FROM workspace.demo_snowday.gold_employee_cdf_type1
ORDER BY id;

-- confirm SDC1 Requirement (no CDF cols)
DESCRIBE TABLE workspace.demo_snowday.gold_employee_cdf_type1;

col_name,data_type,comment
id,int,null
fullname,string,null
sal,double,null
doj,date,null


In [0]:
%sql
-- Step 3 generate a delete event (fresh test)
UPDATE workspace.demo_snowday.landing_employee_cdf
SET sal='1500'
WHERE id='13';

DELETE FROM workspace.demo_snowday.landing_employee_cdf
WHERE id='13';

num_affected_rows
1


In [0]:
%sql
-- Step 4 — Confirm Source CDF produced delete
SELECT id, _change_type, _commit_version
FROM table_changes('workspace.demo_snowday.landing_employee_cdf', 0)
WHERE id='13'
ORDER BY _commit_version;

id,_change_type,_commit_version
13,insert,1
13,update_postimage,2
13,update_preimage,2
13,delete,3


In [0]:
%sql
-- Step 5 — Confirm Bronze captured delete
SELECT id, _change_type, _commit_version
FROM workspace.demo_snowday.bronze_employee_cdf
WHERE id='13'
ORDER BY _commit_version;

id,_change_type,_commit_version
13,insert,4
13,update_postimage,5
13,update_preimage,5
13,update_preimage,6
13,update_postimage,6


In [0]:
%sql
-- step 6 - Validate Gold SCD1 (expected 0 rows because SCD1 correctly deleted row)
SELECT *
FROM workspace.demo_snowday.gold_employee_cdf_type1
WHERE id=13;

id,fullname,sal,doj
13,Bob,275.0,2018-05-25


In [0]:
%sql
-- step 7 - Validate Archive
SELECT
  id, fullname, sal, doj,
  cdf_change_type, cdf_commit_version, cdf_commit_timestamp, archived_at
FROM workspace.demo_snowday.gold_employee_cdf_archive
WHERE id=13
ORDER BY cdf_commit_version DESC;

id,fullname,sal,doj,cdf_change_type,cdf_commit_version,cdf_commit_timestamp,archived_at
